Builds gold_skill_pairs: co-occurrence + lift (Day 15).
Reads:  jobmarket.silver.silver_posting_skills, silver_job_postings
Writes: jobmarket.gold.gold_skill_pairs
Kaggle-only (truncation would under-pair Adzuna), soft skills excluded,
a.skill < b.skill kills self/duplicate pairs. Lift = P(ab)/(P(a)P(b)).

In [0]:
from pyspark.sql import functions as F

sk = spark.table("jobmarket.silver.silver_posting_skills")
sp = spark.table("jobmarket.silver.silver_job_postings")

# Kaggle-only, technical skills only
base = (sk
    .join(sp.filter("source = 'kaggle_backfill'")
            .select("posting_id"), "posting_id")
    .filter(F.col("skill_group") != "soft")
    .select("posting_id", "skill"))

# the probability universe: postings with >=1 technical skill
n_universe = base.select("posting_id").distinct().count()
print(f"Universe: {n_universe} postings")

# per-skill posting counts (for the lift denominator)
skill_counts = (base.groupBy("skill")
    .agg(F.countDistinct("posting_id").alias("n_skill")))

# --- the self-join: one inequality doing three jobs ---
a = base.alias("a")
b = base.alias("b")
pairs = (a.join(b,
        (F.col("a.posting_id") == F.col("b.posting_id")) &
        (F.col("a.skill") < F.col("b.skill")))        # kills self-pairs AND mirror duplicates
    .groupBy(F.col("a.skill").alias("skill_a"),
             F.col("b.skill").alias("skill_b"))
    .agg(F.countDistinct("a.posting_id").alias("co_occurrence")))

print(f"Raw pairs: {pairs.count()}")

# --- lift ---
pairs_lift = (pairs
    .join(skill_counts.withColumnRenamed("skill", "skill_a")
                      .withColumnRenamed("n_skill", "n_a"), "skill_a")
    .join(skill_counts.withColumnRenamed("skill", "skill_b")
                      .withColumnRenamed("n_skill", "n_b"), "skill_b")
    .withColumn("lift",
        F.round((F.col("co_occurrence") * n_universe)
                / (F.col("n_a") * F.col("n_b")), 2))
    .select("skill_a", "skill_b", "co_occurrence", "lift"))

(pairs_lift.write.format("delta").mode("overwrite")
    .saveAsTable("jobmarket.gold.gold_skill_pairs"))

# sanity: top stacks by lift, floored (lift on tiny counts is noise)
spark.table("jobmarket.gold.gold_skill_pairs") \
    .filter("co_occurrence >= 50") \
    .orderBy(F.desc("lift")).limit(15).display()